In [45]:
import nflreadpy as nfl
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
schedules_polars = nfl.load_schedules(seasons=True)
schedules = schedules_polars.to_pandas()
schedules = schedules.rename(columns={'season':'year'})

#Map Columns
#Day of Year (1-365)
schedules['day_of_year'] = pd.to_datetime(schedules['gameday']).dt.dayofyear
#Game Time (in minutes since midnight)
time = pd.to_datetime(schedules['gametime'], format='%H:%M')
schedules['gametime'] =  (time.dt.hour * 60 + time.dt.minute)
#Roof Type (Outdoors, Open, Closed, Dome)
roofs = pd.get_dummies(schedules['roof'], prefix='roof', dtype=int)
#Surfaces
surfaces = pd.get_dummies(schedules['surface'], prefix='surface', dtype=int)
# One-hot team data
teams = pd.get_dummies(
    schedules[['home_team', 'away_team']],
    columns=['home_team', 'away_team'],
    dtype=int
)
schedules = pd.concat([schedules, teams, roofs, surfaces], axis=1)

Index(['game_id', 'year', 'game_type', 'week', 'gameday', 'weekday',
       'gametime', 'away_team', 'away_score', 'home_team',
       ...
       'surface_', 'surface_a_turf', 'surface_astroplay', 'surface_astroturf',
       'surface_dessograss', 'surface_fieldturf', 'surface_grass',
       'surface_grass ', 'surface_matrixturf', 'surface_sportturf'],
      dtype='object', length=131)


In [ ]:

team_features = [
    col for col in schedules.columns
    if col.startswith('home_team_') or col.startswith('away_team_')
]

roof_features = [
    col for col in schedules.columns
    if col.startswith('roof_')
]

surface_features = [
    col for col in schedules.columns
    if col.startswith('surface_')
]

# Regular numerical features
numeric_features = [
    'spread_line',
    'week',
    'year',
    'day_of_year',
    'gametime',
    'away_rest',
    'home_rest',
    'div_game',
    'home_moneyline',
    'away_moneyline'
]

# Combine everything
features = (
    numeric_features
    + roof_features
    + surface_features
    + team_features
)

print("Number of features:", len(features))
print(features)

schedules.to_csv('schedules.csv', index=False)

In [65]:
#Filter Training Data
schedules = schedules[schedules['game_type'] == 'REG'] #Filter for regular season games
schedules = schedules.dropna(subset=["spread_line"])
schedules = schedules.dropna(subset=['gametime'])

training_data = schedules.dropna(subset=['result','home_moneyline','away_moneyline']) #Drop rows with missing result or moneyline data

#Split into train and test sets
X = training_data[features]
y = training_data['result'] > 0 #Convert result from point dif to binary outcome 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Scale features (put everything on scale 0-1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [66]:
model = LogisticRegression(max_iter=10000)
model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=10000)

In [67]:
#Do prediction on test set
y_pred = model.predict(X_test_scaled)

In [69]:
# Test model accuracy results
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Loss", "Win"]))

Model Accuracy: 0.6844

Classification Report:
              precision    recall  f1-score   support

        Loss       0.67      0.55      0.61       448
         Win       0.69      0.79      0.74       566

    accuracy                           0.68      1014
   macro avg       0.68      0.67      0.67      1014
weighted avg       0.68      0.68      0.68      1014



In [55]:

week1 =schedules[(schedules['week'] ==1) & (schedules['year'] == 2026)]
week1 = week1.reset_index(drop=True)
week1_scaled = scaler.transform(week1[features])
week1_probs = model.predict_proba(week1_scaled)[:, 1]
print(week1_probs)


[0.60797579 0.59422738 0.39854738 0.62371345 0.72167807 0.44056486
 0.3739548  0.78164599 0.68955741 0.5521208  0.79521646 0.5889831
 0.51980784 0.67729342 0.38630428 0.57541415]


In [52]:
for index, row in week1.iterrows():
    print(f"Game: {row['away_team']} @ {row['home_team']}, Predicted Win Probability for Home Team: {week1_probs[index]:.4f}")

Game: NE @ SEA, Predicted Win Probability for Home Team: 0.6080
Game: SF @ LA, Predicted Win Probability for Home Team: 0.5942
Game: CHI @ CAR, Predicted Win Probability for Home Team: 0.3985
Game: TB @ CIN, Predicted Win Probability for Home Team: 0.6237
Game: NO @ DET, Predicted Win Probability for Home Team: 0.7217
Game: BUF @ HOU, Predicted Win Probability for Home Team: 0.4406
Game: BAL @ IND, Predicted Win Probability for Home Team: 0.3740
Game: CLE @ JAX, Predicted Win Probability for Home Team: 0.7816
Game: ATL @ PIT, Predicted Win Probability for Home Team: 0.6896
Game: NYJ @ TEN, Predicted Win Probability for Home Team: 0.5521
Game: ARI @ LAC, Predicted Win Probability for Home Team: 0.7952
Game: MIA @ LV, Predicted Win Probability for Home Team: 0.5890
Game: GB @ MIN, Predicted Win Probability for Home Team: 0.5198
Game: WAS @ PHI, Predicted Win Probability for Home Team: 0.6773
Game: DAL @ NYG, Predicted Win Probability for Home Team: 0.3863
Game: DEN @ KC, Predicted Win Pr

In [53]:
#Account for points based on spread. 1pt for favorite, 2 point for underdog. 3 points for 7 pt underdog
#add cols for homeIsFavorite (bool), 7ptUnderdog (bool), evHome (float), evAway (float), bestEV ("Home or Away")
week1_results = pd.DataFrame()
#Favorites
week1_results.insert(0, 'favorite', np.where(
    week1['spread_line'] >= 0, 
    week1['home_team'], 
    week1['away_team'])
)
#Underdogs
week1_results['underdog'] = np.where(
    week1['spread_line'] >= 0, 
    week1['away_team'], 
    week1['home_team'])
#Spread
week1_results['spread'] = np.where(
    week1['home_team'] == week1_results['favorite'],
    week1['spread_line'],
    week1['spread_line'] * -1,
)
#Favorite Win Prob
week1_results['home_win_prob'] = np.where(
    week1['home_team'] == week1_results['favorite'],
    week1_probs,
    1 - week1_probs,
)
#Underdog Win Prob
week1_results['underdog_win_prob'] = 1 - week1_results['home_win_prob']
#EV Home
week1_results['favorite_ev'] = week1_results['home_win_prob'] * 1
#Is Underdog 3pt (7+ point spread)
week1_results['is_3pt_underdog'] = week1_results['spread'] >= 7
#Ev Away
week1_results['underdog_ev'] = np.where(
    week1_results['is_3pt_underdog'],
    week1_results['underdog_win_prob'] * 3,
    week1_results['underdog_win_prob'] * 2
)
#Team Pick (highest EV)
week1_results['pick'] = np.where(
    week1_results['favorite_ev'] >= week1_results['underdog_ev'],
    week1_results['favorite'],
    week1_results['underdog']
)
print(week1_results)

   favorite underdog  spread  home_win_prob  underdog_win_prob  favorite_ev  \
0       SEA       NE     3.0       0.607976           0.392024     0.607976   
1        LA       SF     3.5       0.594227           0.405773     0.594227   
2       CHI      CAR     3.0       0.601453           0.398547     0.601453   
3       CIN       TB     3.5       0.623713           0.376287     0.623713   
4       DET       NO     7.0       0.721678           0.278322     0.721678   
5       BUF      HOU     1.5       0.559435           0.440565     0.559435   
6       BAL      IND     3.5       0.626045           0.373955     0.626045   
7       JAX      CLE     8.5       0.781646           0.218354     0.781646   
8       PIT      ATL     5.5       0.689557           0.310443     0.689557   
9       TEN      NYJ     1.5       0.552121           0.447879     0.552121   
10      LAC      ARI     9.5       0.795216           0.204784     0.795216   
11       LV      MIA     3.0       0.588983         

In [60]:
#Previous Year Comparison: get last years and figure out what score it would have made
YEAR = 2025
previous_year = schedules[schedules['year'] == YEAR]
previous_year = previous_year.reset_index(drop=True)
previous_year_scaled = scaler.transform(previous_year[features])
previous_year_probs = model.predict_proba(previous_year_scaled)[:, 1]

#Account for points based on spread. 1pt for favorite, 2 point for underdog. 3 points for 7 pt underdog
#add cols for homeIsFavorite (bool), 7ptUnderdog (bool), evHome (float), evAway (float), bestEV ("Home or Away")
previous_year_results = pd.DataFrame()
#Favorites
previous_year_results.insert(0, 'favorite', np.where(
    previous_year['spread_line'] >= 0, 
    previous_year['home_team'], 
    previous_year['away_team'])
)
#Underdogs
previous_year_results['underdog'] = np.where(
    previous_year['spread_line'] >= 0, 
    previous_year['away_team'], 
    previous_year['home_team'])
#Spread
previous_year_results['spread'] = np.where(
    previous_year['home_team'] == previous_year_results['favorite'],
    previous_year['spread_line'],
    previous_year['spread_line'] * -1,
)
#Favorite Win Prob
previous_year_results['home_win_prob'] = np.where(
    previous_year['home_team'] == previous_year_results['favorite'],
    previous_year_probs,
    1 - previous_year_probs,
)
#Underdog Win Prob
previous_year_results['underdog_win_prob'] = 1 - previous_year_results['home_win_prob']
#EV Home
previous_year_results['favorite_ev'] = previous_year_results['home_win_prob'] * 1
#Is Underdog 3pt (7+ point spread)
previous_year_results['is_3pt_underdog'] = previous_year_results['spread'] >= 7
#Ev Away
previous_year_results['underdog_ev'] = np.where(
    previous_year_results['is_3pt_underdog'],
    previous_year_results['underdog_win_prob'] * 3,
    previous_year_results['underdog_win_prob'] * 2
)
#Team Pick (highest EV)
previous_year_results['pick'] = np.where(
    previous_year_results['favorite_ev'] >= previous_year_results['underdog_ev'],
    previous_year_results['favorite'],
    previous_year_results['underdog']
)

previous_year_results['actual_winner'] = np.where(
    previous_year['result'] > 0,
    previous_year['home_team'],
    previous_year['away_team']
)

previous_year_points = np.where(
    previous_year_results['pick'] == previous_year_results['actual_winner'],
    np.where(
        previous_year_results['pick'] == previous_year_results['favorite'],
        1,
        np.where(
            previous_year_results['is_3pt_underdog'],
            3,
            2
        )
    ),
    0
)

print(previous_year_results)

print(f"Total Points for {YEAR}: {np.sum(previous_year_points)}")

#what percent were underdogs that won? (winner was not the favorite)
underdog_wins = previous_year_results[previous_year_results['actual_winner'] == previous_year_results['underdog']]
underdog_win_percentage = len(underdog_wins) / len(previous_year_results) * 100
print(f"Percentage of Underdog Wins in {YEAR}: {underdog_win_percentage:.2f}%")
#points if you picked perfectly (winners only)
perfect_points = np.where(
    previous_year_results['actual_winner'] == previous_year_results['favorite'],
    1,
    np.where(
        previous_year_results['is_3pt_underdog'],
        3,
        2
    )
)
print(f"Total Points if Picked Perfectly in {YEAR}: {np.sum(perfect_points)}")

previous_year_results.to_csv('2025.csv', index=False)

    favorite underdog  spread  home_win_prob  underdog_win_prob  favorite_ev  \
0        PHI      DAL     8.5       0.772076           0.227924     0.772076   
1         KC      LAC     3.0       0.628892           0.371108     0.628892   
2         TB      ATL     1.5       0.549450           0.450550     0.549450   
3        CIN      CLE     5.5       0.691640           0.308360     0.691640   
4        IND      MIA     1.5       0.573558           0.426442     0.573558   
..       ...      ...     ...            ...                ...          ...   
280       LA      CHI     3.5       0.603532           0.396468     0.603532   
281       NE      HOU     3.0       0.629312           0.370688     0.629312   
282       NE      DEN     3.5       0.601229           0.398771     0.601229   
283      SEA       LA     2.5       0.589069           0.410931     0.589069   
284      SEA       NE     4.5       0.682311           0.317689     0.682311   

     is_3pt_underdog  underdog_ev pick 